# 02 · Anomaly detectors, side by side

There is no labelled ground truth, so this notebook only shows *what* each detector flags and how much they agree.

In [ ]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
import matplotlib.pyplot as plt
from incubator.load import load_history, load_stress_log, summary
from incubator import features as fe, anomaly as an, thermal_fit as tf, faults as fl
pd.set_option("display.width", 160)

In [ ]:
df = load_history()
ft = fe.feature_table(df)
ft.describe().T

## Detector 1 · rolling z-score of the temperature slope, per heater state

In [ ]:
z = an.rolling_zscore(ft, window=40, threshold=3.0)
an.flagged_episodes(z['flag'])

## Detector 2 · IsolationForest on the feature table

In [ ]:
iso = an.isolation_forest(ft, contamination=0.03)
an.flagged_episodes(iso['flag'])

## Agreement

In [ ]:
an.compare(z['flag'], iso['flag']).as_dict()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df.index, df['temp'], lw=1)
zf = ft.index[z['flag']]; isf = ft.index[iso['flag']]
ax.scatter(zf, df.loc[zf, 'temp'], facecolors='none', edgecolors='C3', label='z-score')
ax.scatter(isf, df.loc[isf, 'temp'] - 0.3, marker='v', color='C1', label='IsolationForest')
ax.legend();

## What the flagged readings look like
Try changing `window`, `threshold` and `contamination` above and re-running; the sets change a lot, which is the point: unsupervised 'anomalies' are a choice of definition, not a fact about the data.

In [ ]:
ft.loc[z['flag']].head(20)

In [ ]:
ft.loc[iso['flag']].head(20)

## Thermal model fits (used by the control-simulation repo)

In [ ]:
first = tf.fit_first_order(df)
first.as_dict()

The heater-lag (second-order) model needs no dead time and halves the error; `figures/model_params.json` holds both parameter sets.

In [ ]:
second = tf.fit_second_order(df)
second.as_dict()

In [ ]:
tf.compare_models(first, second)

Why stop at three parameters? The general two-mass model adds `phi`, the heater body's share of the heat capacity. On this day it is not determined: the free fit runs to the mirror branch and a profile over `phi` is a flat ridge that only trades `phi` against `b` (figure 10).

In [ ]:
tf.fit_two_mass(df).as_dict()

In [ ]:
pd.DataFrame(tf.two_mass_profile(df, phis=(-0.2, 0.0, 0.2, 0.4, 0.6)))

## Detector 3 · drift from the thermal model
The only detector that knows what the heater is supposed to do. Threshold = 99.7th percentile of |drift| on this (clean) day.

In [ ]:
drift = an.model_residual(df, second, threshold_c=0.22)
ax = drift['drift'].plot(figsize=(11, 3))
ax.axhline(0.22, color='C3'); ax.axhline(-0.22, color='C3');
an.flagged_episodes(drift['flag'])

## Detector 4 · sampling-interval watchdog
Not a statistic: it flags the first reading after a gap longer than 60 s. The clean day has exactly one such gap.

In [ ]:
gaps = an.interval_watchdog(df)
gaps.loc[gaps['flag']]

## Scoring on injected faults
The day has no known fault, so `faults.py` injects six kinds (physical ones synthesised with the fitted model) at random positions and scores every detector. `n_trials=5` here to keep it quick; the README table uses 20.

In [ ]:
fault = fl.make_fault('heater_stuck_on', pd.Timestamp('2026-10-03 14:00', tz='UTC'))
faulted, labels = fl.inject(df, fault, second)
ax = faulted['temp'].plot(figsize=(11, 3), label='with fault')
df['temp'].plot(ax=ax, ls=':', label='real'); ax.legend();

In [ ]:
trials = fl.run_trials(df, second, n_trials=5, seed=0)
fl.summarise(trials).round(2)